In [0]:
df = spark.table("bronze_crawling.10000recipe_crawl.ingregient1")
display(df)

In [0]:
from pyspark.sql.functions import col, count

# 중복값 제거된 std_name 목록
df_unique = df.dropDuplicates(['std_name'])
display(df_unique)

# 중복값 제거된 목록 (즉, 중복된 std_name만 추출)
df_duplicates = df.groupBy('std_name').count().filter(col('count') > 1)
display(df_duplicates)

In [0]:
from pyspark.sql.functions import col

# === 비식품 항목 목록 (조리도구/용기/추상어/비식용재료) ===
inedible_list = [
    # 조리도구/주방기구/용기
    '가스렌지', '가스오', '가위한', '거름망', '거즈', '거품기', '거품기수동',
    '고무', '고무줄밴드', '고정용', '광목', '글로브', '그릇', '궁중팬',
    '긴꼬치', '김발', '김밥용김발', '꼬지', '꼬지용', '꼬챙이', '꼬치',
    '꼬치꼬지', '꼬치용꼬지', '꼬치용꼬치', '꼬치용나무',
    '나무', '나무꼬지', '나무꼬치', '나무꼬치산적꼬치', '나무꽂이',
    '나무막', '나무스틱', '나무젓가락', '냄비', '냅킨', '네일',
    '높은틀', '누름돌', '뒤집', '뚜껑', '뚝배기', '뚝배기냄비',
    '대나무', '대나무꼬지', '대나무꼬치', '대나무손잡이꼬지',
    '과도', '과일꽂이', '과일용꼬지', '계량', '계량스푼', '계량컵',
    '믹서기', '믹싱볼', '무스링', '무스틀', '몰드',
    '바통팬', '반죽기', '머핀유산지', '머핀컵', '머핀틀',
    '산적꼬지', '산적꼬치', '산적꽂이', '산적용꼬지', '산적용꼬치',
    '석쇠', '손잡이냄비', '스크래퍼', '스쿱', '스푼',
    '글래드매직랩', '매직랩', '무명실', '면보',
    '디저트컵', '리본', '밀폐', '보틀', '병뚜껑',
    '베주머니', '빵칼', '빼빼로스틱', '사각시트', '쉬폰',
    '스탠드지퍼백', '스타킹', '스펀지', '벚꽃깍지', '별깍지',
    # 비식용 재료/소모품
    '부탄가스', '부품', '빨래건조', '과탄산소다',
    '베이킹소다세척', '베이킹소다세척용',
    # 추상적 용어/수량/단위/메타 정보
    '각각', '각종', '각향', '간재료', '개당', '개조', '곁들임재료',
    '길이', '높이', '두께', '두름',
    '국내산', '국물재료', '국물재료물',
    '꾸미', '꾸이꾸이', '꾸이맨', '끼니',
    '다용도', '다용도무침', '담금', '담백한맛',
    '데코용', '데코용야채', '도시락',
    '만든', '만들기', '미니', '대리',
    '밑간', '반찬', '별로', '부재료',
    '사용', '사이드', '선택재료', '셋트', '속재료', '순간', '시간',
    '1인분', '1차', '1참고', '2분', '2차', '3개', '3개정도',
    '42g', '700', '5L', '25cm',
]

# 비식품 제거
df_edible = df_unique.filter(~col('std_name').isin(inedible_list))

print(f"원본 고유 재료 수: {df_unique.count()}")
print(f"비식품 제거 후 재료 수: {df_edible.count()}")
print(f"제거된 비식품 수: {df_unique.count() - df_edible.count()}")
display(df_edible)

In [0]:
df_edible.write.format("delta").mode("overwrite").saveAsTable("silver.ingredient.ingredient_recipe")